# Recommender System Model Comparison

Comparison of neural recommender models for preference elicitation:
- **One-Hot (LSTM+Attention)**: Paper baseline with learned embeddings
- **Concept (SBERT+LSTM)**: Semantic embeddings via sentence transformers
- **Two-Tower**: Contrastive learning with user/item towers
- **Two-Tower v2**: Separate like/dislike encodings (experimental)
- **LLM Baseline**: GPT-4o-mini for comparison

Key metrics: Loss, Accuracy, NDCG by number of revealed preferences (RL signal).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import json
import pickle
import hashlib
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/movielens')
CACHE_DIR = DATA_DIR / '.cache'
CHECKPOINT_DIR = CACHE_DIR / 'checkpoints'
RESULTS_CACHE = CACHE_DIR / 'notebook_eval_cache.pkl'

N_MOVIES = 100
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data Loading

In [ ]:
# Load MovieLens data
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
movies = pd.read_csv(DATA_DIR / 'movies.csv')

# Top 100 movies by rating count
movie_counts = ratings.groupby('movieId').size().reset_index(name='count')
movies_merged = movies.merge(movie_counts, on='movieId', how='left').fillna(0)
top_movies_df = movies_merged.nlargest(N_MOVIES, 'count')
top_movies = top_movies_df['movieId'].tolist()

movie_to_idx = {mid: i for i, mid in enumerate(top_movies)}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}
movie_titles = {row['movieId']: row['title'] for _, row in movies.iterrows()}

# Filter ratings to top movies
ratings_filtered = ratings[ratings['movieId'].isin(top_movies)].copy()

# Users with >= 10 ratings
user_counts = ratings_filtered.groupby('userId').size()
active_users = user_counts[user_counts >= 10].index.tolist()

# Train/val split
np.random.seed(SEED)
shuffled = active_users.copy()
np.random.shuffle(shuffled)
val_users = shuffled[int(0.8 * len(shuffled)):]
EVAL_USERS = val_users[:300]

print(f"Top {N_MOVIES} movies, {len(active_users):,} active users, {len(EVAL_USERS)} eval users")

In [ ]:
# Find Star Wars movies for sanity checks
sw_movies = [(mid, movie_to_idx[mid], movie_titles[mid]) 
             for mid in top_movies if 'Star Wars' in movie_titles.get(mid, '')]
print("Star Wars movies:", [(idx, t[:40]) for _, idx, t in sw_movies])

## 2. Model Definitions

In [ ]:
class ExtrapolationModel(nn.Module):
    """One-hot LSTM+Attention model (paper baseline)."""
    def __init__(self, n_items):
        super().__init__()
        hidden_dim = n_items // 2
        self.embedding = nn.Embedding(n_items + 1, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim + 3, hidden_dim, 1, batch_first=True)
        self.dense1 = nn.Linear(hidden_dim, n_items)
        self.dense2 = nn.Linear(n_items, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=1, batch_first=True)
        self.output = nn.Linear(hidden_dim * 2, n_items)

    def forward(self, index_input, rating_input):
        x = self.embedding(index_input)
        x = torch.cat((x, rating_input), dim=-1)
        enc, _ = self.lstm(x)
        x = torch.relu(self.dense1(enc))
        x = torch.relu(self.dense2(x))
        att, _ = self.attention(enc, x, x)
        return self.output(torch.cat((x, att), dim=-1))


class ConceptEmbeddingModel(nn.Module):
    """SBERT-based concept embeddings model."""
    def __init__(self, n_items, embedding_dim=384, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or n_items // 2
        self.concept_proj = nn.Linear(embedding_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim + 3, hidden_dim, 1, batch_first=True)
        self.dense1 = nn.Linear(hidden_dim, n_items)
        self.dense2 = nn.Linear(n_items, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=1, batch_first=True)
        self.output = nn.Linear(hidden_dim * 2, n_items)

    def forward(self, item_embeddings, rating_input):
        x = self.concept_proj(item_embeddings)
        x = torch.cat((x, rating_input), dim=-1)
        enc, _ = self.lstm(x)
        x = torch.relu(self.dense1(enc))
        x = torch.relu(self.dense2(x))
        att, _ = self.attention(enc, x, x)
        return self.output(torch.cat((x, att), dim=-1))


class UserTower(nn.Module):
    def __init__(self, state_dim=384, emb_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, emb_dim))
    def forward(self, x):
        return self.network(x)

class ItemTower(nn.Module):
    def __init__(self, state_dim=384, emb_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, emb_dim))
    def forward(self, x):
        return self.network(x)

class TwoTowerModel(nn.Module):
    """Two-tower contrastive model (v1 - combined like/dislike encoding)."""
    def __init__(self, state_dim=384, emb_dim=128):
        super().__init__()
        self.user_tower = UserTower(state_dim, emb_dim)
        self.item_tower = ItemTower(state_dim, emb_dim)

    def get_scores(self, user_state, item_features):
        u = F.normalize(self.user_tower(user_state), dim=-1)
        i = F.normalize(self.item_tower(item_features), dim=-1)
        return torch.matmul(u, i.T)


class UserTowerV2(nn.Module):
    """User tower with separate like/dislike projections."""
    def __init__(self, state_dim=384, proj_dim=128, dropout=0.1):
        super().__init__()
        self.like_proj = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, proj_dim))
        self.dislike_proj = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, proj_dim))
        self.user_head = nn.Sequential(
            nn.Linear(proj_dim * 2, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, proj_dim))

    def forward(self, like_state, dislike_state):
        like_emb = self.like_proj(like_state)
        dislike_emb = self.dislike_proj(dislike_state)
        combined = torch.cat([like_emb, dislike_emb], dim=-1)
        return self.user_head(combined)


class TwoTowerModelV2(nn.Module):
    """Two-tower v2 with SEPARATE like/dislike encodings."""
    def __init__(self, state_dim=384, emb_dim=128):
        super().__init__()
        self.user_tower = UserTowerV2(state_dim, emb_dim)
        self.item_tower = ItemTower(state_dim, emb_dim)

    def get_scores(self, like_state, dislike_state, item_features):
        u = F.normalize(self.user_tower(like_state, dislike_state), dim=-1)
        i = F.normalize(self.item_tower(item_features), dim=-1)
        return torch.matmul(u, i.T)

## 3. Load Trained Models

In [ ]:
models = {}
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer('all-MiniLM-L6-v2')
movie_texts = [movie_titles.get(mid, f"Movie {mid}") for mid in top_movies]
movie_emb = torch.FloatTensor(encoder.encode(movie_texts, show_progress_bar=False))

# One-Hot
if (CHECKPOINT_DIR / 'onehot_paper_config.pt').exists():
    ckpt = torch.load(CHECKPOINT_DIR / 'onehot_paper_config.pt', map_location='cpu', weights_only=False)
    model = ExtrapolationModel(ckpt['n_items'])
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models['onehot'] = {'model': model, 'n_items': ckpt['n_items'], 
                        'epoch': ckpt.get('epochs', 0), 'val_loss': ckpt.get('val_loss', 0)}
    print(f"One-Hot: epoch {ckpt.get('epochs', 0)}, val_loss {ckpt.get('val_loss', 0):.4f}")

# Concept
if (CHECKPOINT_DIR / 'concept_paper_config.pt').exists():
    ckpt = torch.load(CHECKPOINT_DIR / 'concept_paper_config.pt', map_location='cpu', weights_only=False)
    hidden_dim = ckpt['model_state_dict']['concept_proj.bias'].shape[0]
    model = ConceptEmbeddingModel(ckpt['n_items'], 384, hidden_dim)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    emb = np.load(CHECKPOINT_DIR / 'concept_embeddings_paper_config.npy')
    models['concept'] = {'model': model, 'n_items': ckpt['n_items'], 'embeddings': emb,
                         'epoch': ckpt.get('epochs', 0), 'val_loss': ckpt.get('val_loss', 0)}
    print(f"Concept: epoch {ckpt.get('epochs', 0)}, val_loss {ckpt.get('val_loss', 0):.4f}")

# Two-Tower V1
if (CHECKPOINT_DIR / 'two_tower_paper_config.pt').exists():
    ckpt = torch.load(CHECKPOINT_DIR / 'two_tower_paper_config.pt', map_location='cpu', weights_only=False)
    model = TwoTowerModel()
    model.user_tower.load_state_dict(ckpt['user_tower'])
    model.item_tower.load_state_dict(ckpt['item_tower'])
    model.eval()
    models['twotower'] = {'model': model, 'encoder': encoder, 'movie_emb': movie_emb,
                          'epoch': ckpt.get('epoch', 0), 'val_loss': ckpt.get('val_loss', 0)}
    print(f"Two-Tower V1: epoch {ckpt.get('epoch', 0)}, val_loss {ckpt.get('val_loss', 0):.4f}")

# Two-Tower V2 (separate like/dislike encodings)
if (CHECKPOINT_DIR / 'two_tower_v2_paper_config.pt').exists():
    ckpt = torch.load(CHECKPOINT_DIR / 'two_tower_v2_paper_config.pt', map_location='cpu', weights_only=False)
    model = TwoTowerModelV2()
    model.user_tower.load_state_dict(ckpt['user_tower'])
    model.item_tower.load_state_dict(ckpt['item_tower'])
    model.eval()
    models['twotower_v2'] = {'model': model, 'encoder': encoder, 'movie_emb': movie_emb,
                              'epoch': ckpt.get('epoch', 0), 'val_loss': ckpt.get('val_loss', 0)}
    print(f"Two-Tower V2: epoch {ckpt.get('epoch', 0)}, val_loss {ckpt.get('val_loss', 0):.4f}")

print(f"\nLoaded {len(models)} models")

## 4. Evaluation Functions

In [ ]:
def get_user_data(user_id):
    """Get user ground truth and item list."""
    user_df = ratings_filtered[ratings_filtered['userId'] == user_id]
    gt = np.full(N_MOVIES, np.nan)
    items = []
    for _, row in user_df.iterrows():
        if row['movieId'] in movie_to_idx:
            idx = movie_to_idx[row['movieId']]
            gt[idx] = 1.0 if row['rating'] >= 4 else 0.0
            items.append((idx, row['rating'], row['movieId']))
    return gt, items

def calc_metrics(preds, gt, mask=None):
    """Calculate BCE loss, accuracy, NDCG."""
    if mask is None:
        mask = ~np.isnan(gt)
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan
    y_true, y_pred = gt[mask], np.clip(preds[mask], 1e-7, 1-1e-7)
    loss = -np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))
    acc = np.mean((y_pred > 0.5) == y_true)
    # NDCG
    rated_idx = np.where(mask)[0]
    sorted_idx = np.argsort(-preds[rated_idx])[:10]
    rel = gt[rated_idx][sorted_idx]
    dcg = np.sum(rel / np.log2(np.arange(2, len(rel)+2)))
    ideal = np.sort(gt[rated_idx])[::-1][:10]
    idcg = np.sum(ideal / np.log2(np.arange(2, len(ideal)+2)))
    ndcg = dcg / idcg if idcg > 0 else 0
    return loss, acc, ndcg

def predict_onehot(model, n_items, revealed):
    indices = torch.arange(n_items).unsqueeze(0)
    rating_input = torch.zeros(1, n_items, 3)
    rating_input[:, :, 2] = 1
    for idx, r, _ in revealed:
        rating_input[:, idx, 2] = 0
        rating_input[:, idx, 1 if r >= 4 else 0] = 1
    with torch.no_grad():
        return model(indices, rating_input)[:, -1, :N_MOVIES].sigmoid().numpy().flatten()

def predict_concept(model, emb, revealed):
    n_items = emb.shape[0]
    item_emb = torch.FloatTensor(emb).unsqueeze(0)
    rating_input = torch.zeros(1, n_items, 3)
    rating_input[:, :, 2] = 1
    for idx, r, _ in revealed:
        rating_input[:, idx, 2] = 0
        rating_input[:, idx, 1 if r >= 4 else 0] = 1
    with torch.no_grad():
        return model(item_emb, rating_input)[:, -1, :N_MOVIES].sigmoid().numpy().flatten()

def predict_twotower(model, encoder, movie_emb, revealed):
    """Two-Tower V1: combined like/dislike text encoding."""
    liked = [movie_titles.get(mid, "")[:30] for idx, r, mid in revealed if r >= 4][:10]
    disliked = [movie_titles.get(mid, "")[:30] for idx, r, mid in revealed if r < 4][:5]
    parts = []
    if liked: parts.append(f"likes: {', '.join(liked)}")
    if disliked: parts.append(f"dislikes: {', '.join(disliked)}")
    text = ' | '.join(parts) if parts else "no preferences"
    state = torch.FloatTensor(encoder.encode([text], show_progress_bar=False))
    with torch.no_grad():
        scores = model.get_scores(state, movie_emb)
        return torch.sigmoid(scores * 5).numpy().flatten()

def predict_twotower_v2(model, encoder, movie_emb, revealed):
    """Two-Tower V2: separate like/dislike encodings."""
    liked = [movie_titles.get(mid, "")[:30] for idx, r, mid in revealed if r >= 4][:10]
    disliked = [movie_titles.get(mid, "")[:30] for idx, r, mid in revealed if r < 4][:5]
    
    like_text = f"liked: {', '.join(liked)}" if liked else "no liked movies"
    dislike_text = f"disliked: {', '.join(disliked)}" if disliked else "no disliked movies"
    
    like_state = torch.FloatTensor(encoder.encode([like_text], show_progress_bar=False))
    dislike_state = torch.FloatTensor(encoder.encode([dislike_text], show_progress_bar=False))
    
    with torch.no_grad():
        scores = model.get_scores(like_state, dislike_state, movie_emb)
        return torch.sigmoid(scores * 5).numpy().flatten()

## 5. RL Signal Evaluation (Loss by Timestep)

In [ ]:
def run_timestep_eval(model_name, timesteps=[1, 3, 5, 10, 20, 30, 50]):
    """Evaluate model at different timesteps."""
    results = {t: {'loss': [], 'acc': [], 'ndcg': [], 'h_loss': [], 'h_acc': []} for t in timesteps}
    
    for user_id in tqdm(EVAL_USERS, desc=model_name):
        gt, items = get_user_data(user_id)
        if len(items) < 5: continue
        np.random.seed(user_id)
        np.random.shuffle(items)
        
        for t in timesteps:
            revealed = items[:min(t, len(items))]
            revealed_idx = set(idx for idx, _, _ in revealed)
            
            if model_name == 'onehot':
                preds = predict_onehot(models['onehot']['model'], models['onehot']['n_items'], revealed)
            elif model_name == 'concept':
                preds = predict_concept(models['concept']['model'], models['concept']['embeddings'], revealed)
            elif model_name == 'twotower':
                preds = predict_twotower(models['twotower']['model'], models['twotower']['encoder'],
                                         models['twotower']['movie_emb'], revealed)
            elif model_name == 'twotower_v2':
                preds = predict_twotower_v2(models['twotower_v2']['model'], models['twotower_v2']['encoder'],
                                            models['twotower_v2']['movie_emb'], revealed)
            else:
                continue
            
            loss, acc, ndcg = calc_metrics(preds, gt)
            if not np.isnan(loss):
                results[t]['loss'].append(loss)
                results[t]['acc'].append(acc)
                results[t]['ndcg'].append(ndcg)
            
            # Holdout
            h_mask = ~np.isnan(gt) & np.array([i not in revealed_idx for i in range(N_MOVIES)])
            if h_mask.sum() > 0:
                h_loss, h_acc, _ = calc_metrics(preds, gt, h_mask)
                if not np.isnan(h_loss):
                    results[t]['h_loss'].append(h_loss)
                    results[t]['h_acc'].append(h_acc)
    
    return {t: {k: np.mean(v) if v else np.nan for k, v in r.items()} for t, r in results.items()}

In [ ]:
# Run evaluation (with caching)
MODEL_LIST = ['onehot', 'concept', 'twotower', 'twotower_v2']

if RESULTS_CACHE.exists():
    with open(RESULTS_CACHE, 'rb') as f:
        all_results = pickle.load(f)
    print(f"Loaded cached results for: {list(all_results.keys())}")
    # Check if we need to evaluate new models
    for name in MODEL_LIST:
        if name in models and name not in all_results:
            print(f"Evaluating new model: {name}")
            all_results[name] = run_timestep_eval(name)
            with open(RESULTS_CACHE, 'wb') as f:
                pickle.dump(all_results, f)
else:
    all_results = {}
    for name in MODEL_LIST:
        if name in models:
            all_results[name] = run_timestep_eval(name)
    with open(RESULTS_CACHE, 'wb') as f:
        pickle.dump(all_results, f)
    print("Saved results to cache")

In [ ]:
# Results table
timesteps = [1, 3, 5, 10, 20, 30, 50]
for name in MODEL_LIST:
    if name not in all_results: continue
    print(f"\n{name.upper()}:")
    print(f"{'t':<4} {'Loss':<8} {'Acc':<8} {'NDCG':<8} {'H-Loss':<8} {'H-Acc':<8}")
    for t in timesteps:
        r = all_results[name].get(t, {})
        print(f"{t:<4} {r.get('loss',0):.4f}   {r.get('acc',0):.4f}   {r.get('ndcg',0):.4f}   "
              f"{r.get('h_loss',0):.4f}   {r.get('h_acc',0):.4f}")

## 6. Visualization: RL Signal

In [ ]:
# Plot Loss by Timestep
fig = make_subplots(rows=1, cols=2, subplot_titles=['Full Set Loss', 'Holdout Loss'])
colors = {'onehot': '#1f77b4', 'concept': '#ff7f0e', 'twotower': '#2ca02c', 'twotower_v2': '#9467bd'}
timesteps = [1, 3, 5, 10, 20, 30, 50]

for name in MODEL_LIST:
    if name not in all_results: continue
    losses = [all_results[name][t]['loss'] for t in timesteps]
    h_losses = [all_results[name][t]['h_loss'] for t in timesteps]
    fig.add_trace(go.Scatter(x=timesteps, y=losses, name=name, mode='lines+markers',
                             line=dict(color=colors[name])), row=1, col=1)
    fig.add_trace(go.Scatter(x=timesteps, y=h_losses, name=f"{name} (holdout)", 
                             mode='lines+markers', line=dict(color=colors[name], dash='dash'),
                             showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="# Revealed Preferences")
fig.update_yaxes(title_text="BCE Loss", row=1, col=1)
fig.update_yaxes(title_text="BCE Loss", row=1, col=2)
fig.update_layout(height=400, title="RL Signal: Loss Reduction as Preferences Revealed")
fig.show()

In [ ]:
# Plot Accuracy by Timestep
fig = go.Figure()
for name in MODEL_LIST:
    if name not in all_results: continue
    accs = [all_results[name][t]['acc'] for t in timesteps]
    fig.add_trace(go.Scatter(x=timesteps, y=accs, name=name, mode='lines+markers',
                             line=dict(color=colors[name])))

fig.update_layout(xaxis_title="# Revealed Preferences", yaxis_title="Accuracy",
                  title="Accuracy Improvement with More Preferences", height=350)
fig.show()

In [ ]:
# RL Signal Summary
print("RL Signal (1 -> 50 inputs):")
for name in MODEL_LIST:
    if name not in all_results: continue
    loss_1, loss_50 = all_results[name][1]['loss'], all_results[name][50]['loss']
    acc_1, acc_50 = all_results[name][1]['acc'], all_results[name][50]['acc']
    print(f"{name:12} | Loss: {loss_1:.3f} -> {loss_50:.3f} ({loss_1-loss_50:+.3f}) | "
          f"Acc: {acc_1:.3f} -> {acc_50:.3f} ({acc_50-acc_1:+.3f})")

## 7. Sanity Check: Star Wars Correlation

In [ ]:
# Test: Like Episode IV -> Where do other SW movies rank?
# Compare to baseline (mean rating rank)
ep4 = next((m for m in sw_movies if 'IV' in m[2] or 'New Hope' in m[2]), sw_movies[0])
print(f"Input: LIKED '{ep4[2][:50]}'\n")

# Calculate baseline ranks (by mean rating)
mean_ratings = ratings_filtered.groupby('movieId')['rating'].mean()
baseline_ranked = mean_ratings.reindex(top_movies).sort_values(ascending=False)
baseline_rank = {mid: i+1 for i, mid in enumerate(baseline_ranked.index)}

# Get model predictions
model_ranks = {}
for name in MODEL_LIST:
    if name not in models: continue
    revealed = [(ep4[1], 5.0, ep4[0])]
    if name == 'onehot':
        preds = predict_onehot(models['onehot']['model'], models['onehot']['n_items'], revealed)
    elif name == 'concept':
        preds = predict_concept(models['concept']['model'], models['concept']['embeddings'], revealed)
    elif name == 'twotower':
        preds = predict_twotower(models['twotower']['model'], models['twotower']['encoder'],
                                 models['twotower']['movie_emb'], revealed)
    elif name == 'twotower_v2':
        preds = predict_twotower_v2(models['twotower_v2']['model'], models['twotower_v2']['encoder'],
                                    models['twotower_v2']['movie_emb'], revealed)
    ranks = np.argsort(-preds)
    model_ranks[name] = {idx: np.where(ranks == idx)[0][0] + 1 for idx in range(len(preds))}

# Display comparison table
print(f"{'Movie':<35} | {'Base':>5} | {'OH':>4} | {'Con':>4} | {'2T':>4} | {'2Tv2':>5}")
print("-" * 75)
for mid, idx, title in sw_movies:
    if idx == ep4[1]: continue  # Skip input movie
    short_title = title.split('(')[0].strip()[:33]
    base_r = baseline_rank.get(mid, 0)
    oh_r = model_ranks.get('onehot', {}).get(idx, 0)
    con_r = model_ranks.get('concept', {}).get(idx, 0)
    tt_r = model_ranks.get('twotower', {}).get(idx, 0)
    tt2_r = model_ranks.get('twotower_v2', {}).get(idx, 0)
    print(f"{short_title:<35} | {base_r:>5} | {oh_r:>4} | {con_r:>4} | {tt_r:>4} | {tt2_r:>5}")

In [ ]:
# Top 10 movies most affected by LIKED vs DISLIKED Star Wars IV
print(f"Top 10 movies MOST AFFECTED by liking vs disliking Episode IV:\n")

for name in MODEL_LIST:
    if name not in models: continue
    
    liked_rev = [(ep4[1], 5.0, ep4[0])]
    disliked_rev = [(ep4[1], 1.0, ep4[0])]
    
    if name == 'onehot':
        preds_l = predict_onehot(models['onehot']['model'], models['onehot']['n_items'], liked_rev)
        preds_d = predict_onehot(models['onehot']['model'], models['onehot']['n_items'], disliked_rev)
    elif name == 'concept':
        preds_l = predict_concept(models['concept']['model'], models['concept']['embeddings'], liked_rev)
        preds_d = predict_concept(models['concept']['model'], models['concept']['embeddings'], disliked_rev)
    elif name == 'twotower':
        preds_l = predict_twotower(models['twotower']['model'], models['twotower']['encoder'],
                                   models['twotower']['movie_emb'], liked_rev)
        preds_d = predict_twotower(models['twotower']['model'], models['twotower']['encoder'],
                                   models['twotower']['movie_emb'], disliked_rev)
    elif name == 'twotower_v2':
        preds_l = predict_twotower_v2(models['twotower_v2']['model'], models['twotower_v2']['encoder'],
                                      models['twotower_v2']['movie_emb'], liked_rev)
        preds_d = predict_twotower_v2(models['twotower_v2']['model'], models['twotower_v2']['encoder'],
                                      models['twotower_v2']['movie_emb'], disliked_rev)
    
    diff = preds_l - preds_d
    top_affected = np.argsort(-np.abs(diff))[:10]
    
    print(f"{name.upper()}:")
    for idx in top_affected:
        mid = idx_to_movie[idx]
        title = movie_titles.get(mid, f"Movie {mid}")[:40]
        print(f"  {title:<40} | L:{preds_l[idx]:.3f} D:{preds_d[idx]:.3f} delta:{diff[idx]:+.3f}")
    print()

## 8. Two-Tower v2: Separate Like/Dislike Encodings

The original Two-Tower model shows no RL signal. This may be due to encoding likes and dislikes together in one text string. Here we test a variant with separate encodings.

In [ ]:
# Two-Tower V2 comparison note
print("Two-Tower Architecture Comparison:")
print("-" * 50)
print("V1: 'likes: A, B | dislikes: C' -> single SBERT -> user tower")
print("V2: 'liked: A, B' -> SBERT -> like_proj")
print("    'disliked: C' -> SBERT -> dislike_proj")
print("    concat([like_proj, dislike_proj]) -> user_head")
print()
print("Hypothesis: V1 loses polarity when SBERT encodes mixed sentiment.")
print("V2 preserves like/dislike signal through separate projections.")
print()
if 'twotower_v2' in models:
    print(f"V2 loaded: epoch {models['twotower_v2']['epoch']}, val_loss {models['twotower_v2']['val_loss']:.4f}")
else:
    print("V2 checkpoint not found - training in progress?")

## 9. LLM Baseline (GPT-4o-mini)

In [ ]:
import os
import hashlib

LLM_CACHE_DIR = CACHE_DIR / 'llm_responses'
LLM_CACHE_DIR.mkdir(exist_ok=True)

def llm_cache_key(prompt, model):
    return hashlib.sha256(f"{model}:{prompt}".encode()).hexdigest()[:16]

def call_llm_cached(prompt, model="gpt-4o-mini"):
    """Call LLM with caching."""
    key = llm_cache_key(prompt, model)
    cache_file = LLM_CACHE_DIR / f"{key}.json"
    
    if cache_file.exists():
        with open(cache_file) as f:
            return json.load(f).get('response')
    
    try:
        import openai
        client = openai.OpenAI()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )
        result = resp.choices[0].message.content
        with open(cache_file, 'w') as f:
            json.dump({'model': model, 'response': result}, f)
        return result
    except Exception as e:
        print(f"LLM error: {e}")
        return None

def parse_llm_ranking(response, movie_list):
    """Parse LLM response into movie rankings."""
    if not response:
        return []
    rankings = []
    for line in response.split('\n'):
        for movie in movie_list:
            if movie[:20].lower() in line.lower():
                if movie not in rankings:
                    rankings.append(movie)
                break
    return rankings

In [ ]:
# Check for existing LLM results
LLM_RESULTS_FILE = CACHE_DIR / 'llm_experiment_results_v2.json'

if LLM_RESULTS_FILE.exists():
    with open(LLM_RESULTS_FILE) as f:
        llm_results = json.load(f)
    print(f"Loaded cached LLM results: {llm_results.get('n_users', 0)} users")
else:
    print("No cached LLM results. Run LLM evaluation below.")
    llm_results = None

In [ ]:
# LLM Evaluation (skip if already cached)
if llm_results is None:
    N_LLM_USERS = 50
    llm_timesteps = [1, 3, 5, 10]
    llm_results = {'n_users': 0, 'timesteps': llm_timesteps, 'results': {t: [] for t in llm_timesteps}}
    
    movie_list = [movie_titles.get(mid, f"Movie {mid}") for mid in top_movies]
    
    for user_id in tqdm(EVAL_USERS[:N_LLM_USERS], desc="LLM eval"):
        gt, items = get_user_data(user_id)
        if len(items) < 10: continue
        np.random.seed(user_id)
        np.random.shuffle(items)
        llm_results['n_users'] += 1
        
        for t in llm_timesteps:
            revealed = items[:min(t, len(items))]
            liked = [movie_titles.get(mid, "") for idx, r, mid in revealed if r >= 4]
            disliked = [movie_titles.get(mid, "") for idx, r, mid in revealed if r < 4]
            
            prompt = f"""Based on these movie preferences, rank the top 10 movies this user would enjoy:
Liked: {', '.join(liked[:10]) if liked else 'None'}
Disliked: {', '.join(disliked[:5]) if disliked else 'None'}

Movies to rank: {', '.join(movie_list[:50])}

Return ONLY a numbered list of 10 movie titles."""
            
            response = call_llm_cached(prompt)
            rankings = parse_llm_ranking(response, movie_list)
            
            if len(rankings) >= 5:
                # Calculate NDCG
                rel = []
                for movie in rankings[:10]:
                    idx = movie_list.index(movie) if movie in movie_list else -1
                    if idx >= 0 and not np.isnan(gt[idx]):
                        rel.append(gt[idx])
                    else:
                        rel.append(0)
                dcg = sum(r / np.log2(i+2) for i, r in enumerate(rel))
                ideal = sorted([gt[i] for i in range(N_MOVIES) if not np.isnan(gt[i])], reverse=True)[:10]
                idcg = sum(r / np.log2(i+2) for i, r in enumerate(ideal))
                ndcg = dcg / idcg if idcg > 0 else 0
                llm_results['results'][t].append(ndcg)
    
    # Save results
    with open(LLM_RESULTS_FILE, 'w') as f:
        json.dump({**llm_results, 
                   'results': {str(t): {'mean': np.mean(v), 'std': np.std(v)} 
                               for t, v in llm_results['results'].items() if v}}, f, indent=2)
    print(f"Saved LLM results for {llm_results['n_users']} users")

In [ ]:
# Display NDCG for ALL models
print("NDCG by # preferences (all models):\n")
print(f"{'t':<4} | {'OneHot':<8} | {'Concept':<8} | {'2Tower':<8} | {'2Twr_v2':<8} | {'LLM':<8}")
print("-" * 65)

for t in [1, 3, 5, 10]:
    row = f"{t:<4} |"
    
    # Neural models from all_results
    for name in MODEL_LIST:
        if name in all_results and t in all_results[name]:
            ndcg = all_results[name][t].get('ndcg', 0)
            row += f" {ndcg:.4f}  |"
        else:
            row += f" {'N/A':<6} |"
    
    # LLM results
    if llm_results:
        results = llm_results.get('results', {})
        r = results.get(str(t)) or results.get(t) or {}
        if isinstance(r, dict) and 'mean' in r:
            row += f" {r['mean']:.4f}"
        elif isinstance(r, list) and r:
            row += f" {np.mean(r):.4f}"
        else:
            row += f" {'N/A':<6}"
    else:
        row += f" {'N/A':<6}"
    
    print(row)

# Extended timesteps for neural models
print("\nExtended timesteps (neural models only):")
print(f"{'t':<4} | {'OneHot':<8} | {'Concept':<8} | {'2Tower':<8} | {'2Twr_v2':<8}")
print("-" * 55)
for t in [20, 30, 50]:
    row = f"{t:<4} |"
    for name in MODEL_LIST:
        if name in all_results and t in all_results[name]:
            ndcg = all_results[name][t].get('ndcg', 0)
            row += f" {ndcg:.4f}  |"
        else:
            row += f" {'N/A':<6} |"
    print(row)

In [ ]:
# LLM Sanity Check: Star Wars
movie_list = [movie_titles.get(mid, f"Movie {mid}") for mid in top_movies]

print("LLM Sanity Check: Star Wars\n")

prompt_liked = f"""I loved Star Wars: Episode IV - A New Hope. 
Recommend 10 movies I would enjoy from: {', '.join(movie_list[:50])}
Return ONLY a numbered list."""

prompt_disliked = f"""I hated Star Wars: Episode IV - A New Hope.
Recommend 10 movies I would enjoy from: {', '.join(movie_list[:50])}
Return ONLY a numbered list."""

resp_liked = call_llm_cached(prompt_liked)
resp_disliked = call_llm_cached(prompt_disliked)

print("LIKED Star Wars IV:")
print(resp_liked[:500] if resp_liked else "No response")
print("\nDISLIKED Star Wars IV:")
print(resp_disliked[:500] if resp_disliked else "No response")

## 10. Summary & Comparison

In [ ]:
# Plot NDCG comparison (all models)
fig = make_subplots(rows=1, cols=2, subplot_titles=['BCE Loss (neural only)', 'NDCG (all models)'])

colors = {'onehot': '#1f77b4', 'concept': '#ff7f0e', 'twotower': '#2ca02c', 
          'twotower_v2': '#9467bd', 'llm': '#d62728'}

# Left: Loss (neural models only)
for name in MODEL_LIST:
    if name not in all_results: continue
    losses = [all_results[name][t]['loss'] for t in timesteps]
    fig.add_trace(go.Scatter(x=timesteps, y=losses, name=f"{name} (loss)", mode='lines+markers',
                             line=dict(color=colors[name])), row=1, col=1)

# Right: NDCG (all models including LLM)
for name in MODEL_LIST:
    if name not in all_results: continue
    ndcgs = [all_results[name][t].get('ndcg', 0) for t in timesteps]
    fig.add_trace(go.Scatter(x=timesteps, y=ndcgs, name=name, mode='lines+markers',
                             line=dict(color=colors[name])), row=1, col=2)

# Add LLM NDCG
if llm_results:
    llm_ts = [1, 3, 5, 10]
    llm_ndcg = []
    results = llm_results.get('results', {})
    for t in llm_ts:
        r = results.get(str(t)) or results.get(t) or {}
        if isinstance(r, dict) and 'mean' in r:
            llm_ndcg.append(r['mean'])
        elif isinstance(r, list) and r:
            llm_ndcg.append(np.mean(r))
        else:
            llm_ndcg.append(0)
    if any(v > 0 for v in llm_ndcg):
        fig.add_trace(go.Scatter(x=llm_ts, y=llm_ndcg, name='LLM', mode='lines+markers',
                                 line=dict(color=colors['llm'], dash='dot')), row=1, col=2)

fig.update_xaxes(title_text="# Revealed Preferences")
fig.update_yaxes(title_text="BCE Loss", row=1, col=1)
fig.update_yaxes(title_text="NDCG@10", row=1, col=2)
fig.update_layout(height=400, title="Model Comparison: RL Signal")
fig.show()

In [ ]:
# Summary table with all metrics
print("=" * 90)
print("SUMMARY: All Models Comparison")
print("=" * 90)

# Loss comparison
print("\n1. BCE LOSS (lower is better):")
print(f"{'Model':<12} | {'@1':<8} | {'@10':<8} | {'@50':<8} | {'Signal (1->50)':<12}")
print("-" * 65)

for name in MODEL_LIST:
    if name not in all_results: continue
    l1 = all_results[name][1]['loss']
    l10 = all_results[name][10]['loss']
    l50 = all_results[name][50]['loss']
    signal = l1 - l50
    print(f"{name:<12} | {l1:<8.4f} | {l10:<8.4f} | {l50:<8.4f} | {signal:+.4f}")

# NDCG comparison
print("\n2. NDCG (higher is better):")
print(f"{'Model':<12} | {'@1':<8} | {'@10':<8} | {'@50':<8} | {'Signal (1->50)':<12}")
print("-" * 65)

for name in MODEL_LIST:
    if name not in all_results: continue
    n1 = all_results[name][1].get('ndcg', 0)
    n10 = all_results[name][10].get('ndcg', 0)
    n50 = all_results[name][50].get('ndcg', 0)
    signal = n50 - n1
    print(f"{name:<12} | {n1:<8.4f} | {n10:<8.4f} | {n50:<8.4f} | {signal:+.4f}")

# LLM row
if llm_results:
    results = llm_results.get('results', {})
    r1 = results.get('1') or results.get(1) or {}
    r10 = results.get('10') or results.get(10) or {}
    n1 = r1.get('mean', 0) if isinstance(r1, dict) else (np.mean(r1) if r1 else 0)
    n10 = r10.get('mean', 0) if isinstance(r10, dict) else (np.mean(r10) if r10 else 0)
    signal = n10 - n1
    print(f"{'LLM':<12} | {n1:<8.4f} | {n10:<8.4f} | {'N/A':<8} | {signal:+.4f} (1->10)")

# Accuracy comparison
print("\n3. ACCURACY (higher is better):")
print(f"{'Model':<12} | {'@1':<8} | {'@10':<8} | {'@50':<8} | {'Signal (1->50)':<12}")
print("-" * 65)

for name in MODEL_LIST:
    if name not in all_results: continue
    a1 = all_results[name][1]['acc']
    a10 = all_results[name][10]['acc']
    a50 = all_results[name][50]['acc']
    signal = a50 - a1
    print(f"{name:<12} | {a1:<8.4f} | {a10:<8.4f} | {a50:<8.4f} | {signal:+.4f}")

print("\n" + "=" * 90)
print("KEY FINDINGS:")
print("-" * 90)
print("- One-Hot: Strong RL signal across all metrics")
print("- Concept: Moderate signal, semantic embeddings help with correlation")
print("- Two-Tower V1: Weak/no signal - combined text encoding loses polarity")
print("- Two-Tower V2: Separate like/dislike encodings - hypothesis: should show signal")
print("- LLM: Shows signal via implicit movie knowledge (limited timesteps tested)")